In [2]:
import os
import zipfile
from itertools import chain

import pandas as pd

In [3]:
data_path           = '../data'
join_lb_gt_path     = f'{data_path}/lakebench_gt/opendata_join_ground_truth.csv'
union_lb_gt_path    = f'{data_path}/lakebench_gt/opendata_union_ground_truth.csv'

## JOIN

In [4]:
lb_join_gt = pd.read_csv(join_lb_gt_path)
lb_join_gt

,query_table,candidate_table,query_column,candidate_column
0,CAN_CSV0000000000001724__13.csv,CAN_CSV0000000000001787__9.csv,RentLocation,RentLocation
1,UK_CSV0000000000003896__7.csv,UK_CSV0000000000002010__11.csv,Post-Op Q Anxiety,Post-Op Q Anxiety
2,UK_CSV0000000000003896__7.csv,UK_CSV0000000000002010__11.csv,Post-Op Q Wound,Post-Op Q Wound
3,USA_CSV0000000000037101__16.csv,USA_CSV0000000000033847__23.csv,STD_VOLUME_GROUP,STD_VOLUME_GROUP
4,USA_CSV0000000000037101__16.csv,USA_CSV0000000000033847__23.csv,ROUTE_SIGNING,ROUTE_SIGNING
...,...,...,...,...
42558,CAN_CSV0000000000013788.csv,CAN_CSV0000000000001071.csv,"ï»¿""REF_DATE""","ï»¿""REF_DATE"""
42559,CAN_CSV0000000000013788.csv,CAN_CSV0000000000000660.csv,"ï»¿""REF_DATE""","ï»¿""REF_DATE"""
42560,CAN_CSV0000000000013788.csv,CAN_CSV0000000000013712.csv,"ï»¿""REF_DATE""","ï»¿""REF_DATE"""
42561,CAN_CSV0000000000013788.csv,CAN_CSV0000000000000857.csv,"ï»¿""REF_DATE""","ï»¿""REF_DATE"""


### Are present all the tables from the Ground Truth?

Answer: YES, all the tables listed into the JOIN Ground Truth are also contained into the relative country dataset

In [5]:
all_join_table_ids = set(filter(lambda s: '__' not in s, chain(*lb_join_gt.values.tolist())))
len(all_join_table_ids)

2508

In [6]:
union_table_ids_by_country = {
    country: {s for s in all_join_table_ids if country in s}
    for country in ['SG', 'USA', 'UK', 'CAN']
}

In [7]:
for country, tables in union_table_ids_by_country.items():
    print(f'{country=}, {len(tables)=}')

country='SG', len(tables)=50
country='USA', len(tables)=334
country='UK', len(tables)=103
country='CAN', len(tables)=961


In [8]:
for country, tables in union_table_ids_by_country.items():
    with zipfile.ZipFile(f'{data_path}/datasets/datasets_{country}.zip') as z:
        names = {f.removeprefix(f'datasets_{country}/') for f in z.namelist()}
        names.remove('')

        diff = tables.difference(names)
        print(f'{country=}, {len(diff)=}')

country='SG', len(diff)=0
country='USA', len(diff)=0
country='UK', len(diff)=0
country='CAN', len(diff)=0


### How many Ground Truth pairs have different names for query and candidate columns?

Answer: None

In [9]:
lb_join_gt[lb_join_gt['query_column'] != lb_join_gt['candidate_column']]

,query_table,candidate_table,query_column,candidate_column


### There are some identical query and candidate tables?

Answer: the 4% of the total JOIN pairs into the Ground Truth is actually a no-sense join on the same column of the same table

In [10]:
lb_join_gt[lb_join_gt['query_table'] == lb_join_gt['candidate_table']].shape[0] * 100 / lb_join_gt.shape[0]

4.010525573855227

In [11]:
lb_join_gt[(lb_join_gt['query_table'] == lb_join_gt['candidate_table']) & (lb_join_gt['query_column'] != lb_join_gt['candidate_column'])]

,query_table,candidate_table,query_column,candidate_column


## UNION

In [12]:
lb_union_gt = pd.read_csv(union_lb_gt_path)
lb_union_gt

,query_table,candidate_table
0,CAN_CSV0000000000000474.csv,CAN_CSV0000000000004972.csv
1,CAN_CSV0000000000000474.csv,CAN_CSV0000000000013001.csv
2,CAN_CSV0000000000000474.csv,CAN_CSV0000000000002292.csv
3,CAN_CSV0000000000000474.csv,CAN_CSV0000000000027218.csv
4,CAN_CSV0000000000000474.csv,CAN_CSV0000000000005832.csv
...,...,...
49510,USA_CSV0000000000037302.csv,USA_CSV0000000000037302.csv
49511,USA_CSV0000000000037322.csv,USA_CSV0000000000037327.csv
49512,USA_CSV0000000000037322.csv,USA_CSV0000000000037322.csv
49513,USA_CSV0000000000037327.csv,USA_CSV0000000000037327.csv


### There are pairs of identical tables?

Answer: YES, the 5% (2688) pairs of tables from the Ground Truth is no-sense, since it is references the same identical table

In [13]:
lb_union_gt[lb_union_gt['query_table'] == lb_union_gt['candidate_table']].shape[0] * 100 / lb_union_gt.shape[0]

5.428657982429566

### All the tables are present into the provided dataset?

Filtering the slices

Answer: YES, it seems that all the tables in the UNION Ground Truth are present into the relative country Open Data

In [14]:
all_union_table_ids = set(filter(lambda s: '__' not in s, chain(*lb_union_gt.values.tolist())))
len(all_union_table_ids)

3784

In [15]:
union_table_ids_by_country = {
    country: {s for s in all_union_table_ids if country in s}
    for country in ['SG', 'USA', 'UK', 'CAN']
}

In [16]:
for country, tables in union_table_ids_by_country.items():
    print(f'{country=}, {len(tables)=}')

country='SG', len(tables)=355
country='USA', len(tables)=700
country='UK', len(tables)=234
country='CAN', len(tables)=2495


In [17]:
for country, tables in union_table_ids_by_country.items():
    with zipfile.ZipFile(f'{data_path}/datasets/datasets_{country}.zip') as z:
        names = {f.removeprefix(f'datasets_{country}/') for f in z.namelist()}
        names.remove('')

        diff = tables.difference(names)
        print(f'{country=}, {len(diff)=}')


country='SG', len(diff)=0
country='USA', len(diff)=0
country='UK', len(diff)=0
country='CAN', len(diff)=0
